# HackerTube 소개: Karpathy가 Claude Opus 5.5와 Claude Code로 해커뉴스 댓글 속 추천 영상 6천여 개를 모아 2시간 만에 만든 YouTube식 사이트, 만든 과정과 공개 데이터로 나만의 버전 만드는 법

위에서부터 차례로 실행해 보세요.

### 내보내기 zip 받기

In [ ]:
%%bash
mkdir -p /content/ht && cd /content/ht
curl -sL -O https://hackertube.org/download/hackertube-data.zip
python3 -c "import zipfile;zipfile.ZipFile('hackertube-data.zip').extractall('data')"
ls -l data/hackertube-data

### 영상 수·주제 수·필드 확인

In [ ]:
import json,collections
d='/content/ht/data/hackertube-data'
v=[json.loads(l) for l in open(d+'/videos.jsonl')]
print('videos',len(v)); print('fields',list(v[0].keys()))
t=json.load(open(d+'/taxonomy.json'))
cats=[c for a in t['areas'] for c in a['categories']]
print('areas',len(t['areas']),'categories',len(cats))
print('mentions',sum(1 for _ in open(d+'/mentions.jsonl')))
print('mention fields',list(json.loads(open(d+'/mentions.jsonl').readline()).keys()))
c=collections.Counter(x['id'] for r in v for x in r['categories'])
print(c.most_common(12))

### 주제 하나 걸러 정렬

In [ ]:
CAT='llms-generative-ai'
sel=[r for r in v if any(c['id']==CAT for c in r['categories']) and r['shelf_life']!='news' and r['hn']['recommenders']>0]
sel.sort(key=lambda r:(-r['score'],-r['hn']['recommenders']))
print('after filter:',len(sel))
for r in sel[:10]:
    print(f"{r['score']:5.1f} rec={r['hn']['recommenders']:3d} {r['hn']['first_year']}-{r['hn']['last_year']} {r['channel'][:22]:22s} {r['title'][:60]}")

### 썸네일 카드 HTML 만들고 보기

In [ ]:
import html
from IPython.display import HTML
top=sel[:24]
cards=''.join(f'''<a href="{r['url']}" target="_blank" style="width:23%;display:inline-block;vertical-align:top;margin:1%;font:13px sans-serif;text-decoration:none;color:#111"><img src="https://img.youtube.com/vi/{r['vid']}/hqdefault.jpg" style="width:100%"><b>{html.escape(r['title'])}</b><br>{html.escape(r['channel'])}<br><span style="color:#ff6600">추천 {r['hn']['recommenders']}명 · {r['hn']['first_year']}–{r['hn']['last_year']}</span></a>''' for r in top)
open('my_hackertube.html','w').write('<meta charset=utf-8>'+cards)
display(HTML(cards))

### HF 월 1개 parquet에서 YouTube ID 뽑기

In [ ]:
%%bash
pip install -q duckdb
mkdir -p /content/hn && curl -sL -o /content/hn/2024-11.parquet https://huggingface.co/datasets/open-index/hacker-news/resolve/main/data/2024/2024-11.parquet
python3 - <<'E'
import duckdb,re,html,collections
f='/content/hn/2024-11.parquet'
rows=duckdb.sql(f"select id,text from '{f}' where type=2 and text ilike '%youtu%'").fetchall()
pat=re.compile(r'(?:youtube\.com/(?:watch\?(?:[^"\s<]*?&)?v=|embed/|shorts/)|youtu\.be/)([A-Za-z0-9_-]{11})')
ids=[m for _,tx in rows for m in pat.findall(html.unescape(tx or ''))]
c=collections.Counter(ids)
print('댓글',len(rows),'ID',len(ids),'고유',len(c)); print(c.most_common(5))
E